## Configuracion de esta corrida

In [ ]:
INFERIR_SRC = '#!/usr/bin/env python\n"""Referencia para la app: corre el .tflite de MicaTB (G-bacilli) con el mismo preprocesado,\ndecodificacion y NMS que debe hacer la app. Dependencias: numpy, opencv-python y ai-edge-litert\n(o tflite-runtime / tensorflow). No usa ultralytics.\n\nUso: python inferir_tflite.py --model ecoli_gb_E3s43_fp16.tflite --config app_config.json --image campo.jpg [--definicion estricta|amplia]\nSalida (JSON): conteo y cajas [x1,y1,x2,y2] en pixeles de la imagen original, con clase y confianza.\n"""\nimport argparse, json\nimport cv2\nimport numpy as np\n\n\ndef cargar_interprete(path):\n    try:\n        from ai_edge_litert.interpreter import Interpreter\n    except ImportError:\n        try:\n            from tflite_runtime.interpreter import Interpreter\n        except ImportError:\n            from tensorflow.lite import Interpreter\n    it = Interpreter(model_path=path); it.allocate_tensors()\n    return it\n\n\ndef letterbox(bgr, size, fill=114):\n    """Escala al lado mayor = size y rellena con gris centrado (igual que ultralytics, auto=False)."""\n    h, w = bgr.shape[:2]\n    r = min(size / h, size / w)\n    nw, nh = round(w * r), round(h * r)\n    dw, dh = (size - nw) / 2, (size - nh) / 2\n    if (w, h) != (nw, nh):\n        bgr = cv2.resize(bgr, (nw, nh), interpolation=cv2.INTER_LINEAR)\n    top, bottom = round(dh - 0.1), round(dh + 0.1)\n    left, right = round(dw - 0.1), round(dw + 0.1)\n    out = cv2.copyMakeBorder(bgr, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(fill, fill, fill))\n    return out, r, left, top\n\n\ndef preprocesar(bgr, cfg):\n    """Letterbox a `contenido` px y borde gris de (tamano - contenido)/2 px por lado -> entrada tamano x tamano."""\n    size = cfg["entrada"]["tamano"]; cont = cfg["entrada"].get("contenido", size); borde = (size - cont) // 2\n    img, r, left, top = letterbox(bgr, cont)\n    if borde:\n        img = cv2.copyMakeBorder(img, borde, borde, borde, borde, cv2.BORDER_CONSTANT, value=(114, 114, 114))\n    x = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0   # RGB, 0..1\n    return np.transpose(x, (2, 0, 1))[None], r, left + borde, top + borde   # NCHW\n\n\ndef nms_por_clase(boxes, scores, classes, iou_thr):\n    """NMS greedy independiente por clase (suprime si IoU > umbral), como torchvision.batched_nms."""\n    keep = []\n    for c in np.unique(classes):\n        idx = np.where(classes == c)[0]; idx = idx[np.argsort(-scores[idx])]\n        while len(idx):\n            i = idx[0]; keep.append(i)\n            if len(idx) == 1: break\n            rest = idx[1:]\n            xx1 = np.maximum(boxes[i, 0], boxes[rest, 0]); yy1 = np.maximum(boxes[i, 1], boxes[rest, 1])\n            xx2 = np.minimum(boxes[i, 2], boxes[rest, 2]); yy2 = np.minimum(boxes[i, 3], boxes[rest, 3])\n            inter = np.clip(xx2 - xx1, 0, None) * np.clip(yy2 - yy1, 0, None)\n            a1 = (boxes[i, 2] - boxes[i, 0]) * (boxes[i, 3] - boxes[i, 1])\n            a2 = (boxes[rest, 2] - boxes[rest, 0]) * (boxes[rest, 3] - boxes[rest, 1])\n            idx = rest[inter / (a1 + a2 - inter + 1e-9) <= iou_thr]\n    return np.array(keep, dtype=int)\n\n\ndef detectar(it, bgr, cfg, definicion="estricta"):\n    size = cfg["entrada"]["tamano"]; H, W = bgr.shape[:2]\n    x, r, left, top = preprocesar(bgr, cfg)\n    i0, o0 = it.get_input_details()[0], it.get_output_details()[0]\n    it.set_tensor(i0["index"], x); it.invoke()\n    y = it.get_tensor(o0["index"])[0]                                   # [4 + n_clases, n_anclas]\n    cx, cy, w, h = y[0] * size, y[1] * size, y[2] * size, y[3] * size   # cajas normalizadas 0..1 -> px de entrada\n    boxes = np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], 1)\n    scores_all = y[4:].T; cls = scores_all.argmax(1); conf = scores_all.max(1)\n    d = cfg["definiciones"][definicion]\n    m = conf > d["conf"]; boxes, conf, cls = boxes[m], conf[m], cls[m]\n    k = nms_por_clase(boxes, conf, cls, d["iou_nms"])[: cfg["max_det"]]\n    boxes, conf, cls = boxes[k], conf[k], cls[k]\n    boxes[:, [0, 2]] = (boxes[:, [0, 2]] - left) / r; boxes[:, [1, 3]] = (boxes[:, [1, 3]] - top) / r   # deshacer letterbox\n    boxes[:, [0, 2]] = boxes[:, [0, 2]].clip(0, W); boxes[:, [1, 3]] = boxes[:, [1, 3]].clip(0, H)\n    clases_contadas = d["clases_contadas"]\n    contar = np.isin(cls, clases_contadas)\n    return dict(conteo=int(contar.sum()),\n                detecciones=[dict(clase=cfg["clases"][int(c)], conf=round(float(s), 4), caja=[round(float(v), 1) for v in b])\n                             for b, s, c in zip(boxes, conf, cls)])\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser(); ap.add_argument("--model", required=True); ap.add_argument("--config", required=True)\n    ap.add_argument("--image", required=True); ap.add_argument("--definicion", default="estricta", choices=["estricta", "amplia"])\n    a = ap.parse_args(); cfg = json.load(open(a.config, encoding="utf8"))\n    img = cv2.imread(a.image); assert img is not None, "no se pudo leer la imagen"\n    print(json.dumps(detectar(cargar_interprete(a.model), img, cfg, a.definicion), indent=1, ensure_ascii=False))\n'

## Funciones comunes (eclib)

In [ ]:
import os, json, time, shutil, subprocess, sys
from pathlib import Path
import numpy as np, pandas as pd, cv2

CLASS_NAMES = ["G-cocci", "G+cocci", "G-bacilli", "G+bacilli"]
GB, GPB = 2, 3          # ids de G-bacilli y G+bacilli
NMS_IOU = 0.7

def prepare_ds(src_root, dst_root="/kaggle/temp/yolo_ds", with_bg=False):
    """Copia el dataset a un lugar escribible y escribe el yaml con rutas reales."""
    src_root, dst_root = Path(src_root), Path(dst_root)
    if not dst_root.exists(): shutil.copytree(src_root, dst_root)
    names = "\n".join(f"  {i}: {n}" for i, n in enumerate(CLASS_NAMES))
    tr = "[images/train, images/bg_train]" if with_bg else "images/train"
    y = dst_root / ("data_bg.yaml" if with_bg else "data_base.yaml")
    y.write_text(f"path: {dst_root}\ntrain: {tr}\nval: images/val\ntest: images/test\nnames:\n{names}\n")
    return dst_root, y

def read_gt(label_path):
    a = np.loadtxt(label_path, ndmin=2) if Path(label_path).stat().st_size else np.zeros((0, 5))
    c = a[:, 0].astype(int); xc, yc, w, h = a[:, 1:5].T
    return c, np.stack([xc-w/2, yc-h/2, xc+w/2, yc+h/2], 1)

def predict_dets(model, img_paths, imgsz, conf=0.01, iou=NMS_IOU, batch=32, device=None, max_det=1000):
    """Detecciones de baja confianza; el barrido de umbral se hace despues sobre este cache."""
    out = {}
    paths = [str(p) for p in img_paths]
    for i in range(0, len(paths), batch):
        for p, r in zip(paths[i:i+batch], model.predict(paths[i:i+batch], imgsz=imgsz, conf=conf, iou=iou, verbose=False, device=device, max_det=max_det)):
            h, w = r.orig_shape
            b = r.boxes
            out[Path(p).stem] = (b.cls.cpu().numpy().astype(int), b.conf.cpu().numpy(), b.xyxy.cpu().numpy() / np.array([w, h, w, h]))
    return out

def box_iou(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    iw = np.clip(np.minimum(a[:, None, 2], b[None, :, 2]) - np.maximum(a[:, None, 0], b[None, :, 0]), 0, None)
    ih = np.clip(np.minimum(a[:, None, 3], b[None, :, 3]) - np.maximum(a[:, None, 1], b[None, :, 1]), 0, None)
    i = iw * ih; ua = ((a[:, 2]-a[:, 0])*(a[:, 3]-a[:, 1]))[:, None] + ((b[:, 2]-b[:, 0])*(b[:, 3]-b[:, 1]))[None] - i
    return i / np.maximum(ua, 1e-9)

def load_gts(ds_root, split):
    return {p.stem: read_gt(p) for p in sorted((Path(ds_root) / "labels" / split).glob("*.txt"))}

def counts(dets, gts, conf, classes):
    """Conteos por imagen (gt, pred) de las clases dadas."""
    rows = []
    for k, (gc, _) in gts.items():
        c, s, _b = dets[k]
        rows.append((int(np.isin(gc, classes).sum()), int((np.isin(c, classes) & (s >= conf)).sum())))
    return np.array(rows)

def count_metrics(dets, gts, conf, classes):
    a = counts(dets, gts, conf, classes); e = a[:, 1] - a[:, 0]
    return dict(mae=float(np.abs(e).mean()), bias=float(e.mean()), exact=float((e == 0).mean()),
                gt_total=int(a[:, 0].sum()), pred_total=int(a[:, 1].sum()))

def boot_ci(dets, gts, conf, classes, n=1000, seed=0):
    a = counts(dets, gts, conf, classes); e = np.abs(a[:, 1] - a[:, 0]); rng = np.random.default_rng(seed)
    m = [e[rng.integers(0, len(e), len(e))].mean() for _ in range(n)]
    return [float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))]

def boot_ci_groups(dets, gts, conf, classes, group_of, n=1000, seed=0, stat="mae"):
    """IC 95 % remuestreando GRUPOS de origen (group_of: stem -> grupo), no imagenes."""
    keys = list(gts); a = counts(dets, gts, conf, classes); e = np.abs(a[:, 1] - a[:, 0])
    g = np.array([group_of[k] for k in keys]); ug, inv = np.unique(g, return_inverse=True)
    sums = np.bincount(inv, weights=e, minlength=len(ug)); cnt = np.bincount(inv, minlength=len(ug)); rng = np.random.default_rng(seed)
    m = []
    for _ in range(n):
        ix = rng.integers(0, len(ug), len(ug)); m.append(sums[ix].sum() / cnt[ix].sum())
    return [float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))]

def match_stats(dets, gts, conf, cls=GB, thr=0.5):
    """TP/FP/FN de la clase cls (IoU>=thr, 1 a 1 greedy) y confusiones con la otra clase de bacilos."""
    tp = fp = fn = conf_to_gpb = 0
    for k, (gc, gbx) in gts.items():
        c, s, b = dets[k]; m = s >= conf
        pc, pb = c[m], b[m]
        g = gc == cls; gb_ = gbx[g]; p = pb[pc == cls]
        iou = box_iou(p, gb_); used_p, used_g = set(), set()
        for v, i, j in sorted(((iou[i, j], i, j) for i in range(len(p)) for j in range(len(gb_))), reverse=True):
            if v < thr: break
            if i in used_p or j in used_g: continue
            used_p.add(i); used_g.add(j)
        t = len(used_p); tp += t; fp += len(p) - t; fn += len(gb_) - t
        other = GPB if cls == GB else GB
        un = [j for j in range(len(gb_)) if j not in used_g]
        if un:
            q = box_iou(pb[pc == other], gb_[un]); conf_to_gpb += int((q >= thr).any(0).sum()) if q.size else 0
    P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
    return dict(tp=tp, fp=fp, fn=fn, precision=P, recall=R, f1=2*P*R/max(P+R, 1e-9), gt_clase_predicha_como_otra=conf_to_gpb)

def sweep(dets, gts, classes, confs=np.round(np.arange(0.05, 0.81, 0.05), 2)):
    rows = []
    for c in confs:
        r = dict(conf=float(c), **count_metrics(dets, gts, c, classes))
        r.update({"f1_Gb": match_stats(dets, gts, c)["f1"]}); rows.append(r)
    return pd.DataFrame(rows)

def bg_fp(dets_bg, conf, classes=(GB,)):
    n = np.array([int((np.isin(c, classes) & (s >= conf)).sum()) for c, s, _ in dets_bg.values()])
    return dict(n_imgs=len(n), fp_por_imagen=float(n.mean()), imgs_con_fp=float((n > 0).mean()))

def cpu_latency_ms(model, img, imgsz, n=100, warm=10):
    for _ in range(warm): model.predict(img, imgsz=imgsz, device="cpu", verbose=False)
    t = []
    for _ in range(n):
        t0 = time.perf_counter(); model.predict(img, imgsz=imgsz, device="cpu", verbose=False); t.append((time.perf_counter()-t0)*1000)
    return float(np.median(t))

def ap_table(m):
    """mAP50 y mAP50-95 por clase desde un objeto DetMetrics de ultralytics."""
    idx = list(m.box.ap_class_index)
    return {CLASS_NAMES[c]: dict(ap50=float(m.box.ap50[i]), ap=float(m.box.ap[i])) for i, c in enumerate(idx)}

#!/usr/bin/env python
"""Referencia para la app: corre el .tflite de MicaTB (G-bacilli) con el mismo preprocesado,
decodificacion y NMS que debe hacer la app. Dependencias: numpy, opencv-python y ai-edge-litert
(o tflite-runtime / tensorflow). No usa ultralytics.

Uso: python inferir_tflite.py --model ecoli_gb_E3s43_fp16.tflite --config app_config.json --image campo.jpg [--definicion estricta|amplia]
Salida (JSON): conteo y cajas [x1,y1,x2,y2] en pixeles de la imagen original, con clase y confianza.
"""
import argparse, json
import cv2
import numpy as np


def cargar_interprete(path):
    try:
        from ai_edge_litert.interpreter import Interpreter
    except ImportError:
        try:
            from tflite_runtime.interpreter import Interpreter
        except ImportError:
            from tensorflow.lite import Interpreter
    it = Interpreter(model_path=path); it.allocate_tensors()
    return it


def letterbox(bgr, size, fill=114):
    """Escala al lado mayor = size y rellena con gris centrado (igual que ultralytics, auto=False)."""
    h, w = bgr.shape[:2]
    r = min(size / h, size / w)
    nw, nh = round(w * r), round(h * r)
    dw, dh = (size - nw) / 2, (size - nh) / 2
    if (w, h) != (nw, nh):
        bgr = cv2.resize(bgr, (nw, nh), interpolation=cv2.INTER_LINEAR)
    top, bottom = round(dh - 0.1), round(dh + 0.1)
    left, right = round(dw - 0.1), round(dw + 0.1)
    out = cv2.copyMakeBorder(bgr, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(fill, fill, fill))
    return out, r, left, top


def preprocesar(bgr, cfg):
    """Letterbox a `contenido` px y borde gris de (tamano - contenido)/2 px por lado -> entrada tamano x tamano."""
    size = cfg["entrada"]["tamano"]; cont = cfg["entrada"].get("contenido", size); borde = (size - cont) // 2
    img, r, left, top = letterbox(bgr, cont)
    if borde:
        img = cv2.copyMakeBorder(img, borde, borde, borde, borde, cv2.BORDER_CONSTANT, value=(114, 114, 114))
    x = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0   # RGB, 0..1
    return np.transpose(x, (2, 0, 1))[None], r, left + borde, top + borde   # NCHW


def nms_por_clase(boxes, scores, classes, iou_thr):
    """NMS greedy independiente por clase (suprime si IoU > umbral), como torchvision.batched_nms."""
    keep = []
    for c in np.unique(classes):
        idx = np.where(classes == c)[0]; idx = idx[np.argsort(-scores[idx])]
        while len(idx):
            i = idx[0]; keep.append(i)
            if len(idx) == 1: break
            rest = idx[1:]
            xx1 = np.maximum(boxes[i, 0], boxes[rest, 0]); yy1 = np.maximum(boxes[i, 1], boxes[rest, 1])
            xx2 = np.minimum(boxes[i, 2], boxes[rest, 2]); yy2 = np.minimum(boxes[i, 3], boxes[rest, 3])
            inter = np.clip(xx2 - xx1, 0, None) * np.clip(yy2 - yy1, 0, None)
            a1 = (boxes[i, 2] - boxes[i, 0]) * (boxes[i, 3] - boxes[i, 1])
            a2 = (boxes[rest, 2] - boxes[rest, 0]) * (boxes[rest, 3] - boxes[rest, 1])
            idx = rest[inter / (a1 + a2 - inter + 1e-9) <= iou_thr]
    return np.array(keep, dtype=int)


def detectar(it, bgr, cfg, definicion="estricta"):
    size = cfg["entrada"]["tamano"]; H, W = bgr.shape[:2]
    x, r, left, top = preprocesar(bgr, cfg)
    i0, o0 = it.get_input_details()[0], it.get_output_details()[0]
    it.set_tensor(i0["index"], x); it.invoke()
    y = it.get_tensor(o0["index"])[0]                                   # [4 + n_clases, n_anclas]
    cx, cy, w, h = y[0] * size, y[1] * size, y[2] * size, y[3] * size   # cajas normalizadas 0..1 -> px de entrada
    boxes = np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], 1)
    scores_all = y[4:].T; cls = scores_all.argmax(1); conf = scores_all.max(1)
    d = cfg["definiciones"][definicion]
    m = conf > d["conf"]; boxes, conf, cls = boxes[m], conf[m], cls[m]
    k = nms_por_clase(boxes, conf, cls, d["iou_nms"])[: cfg["max_det"]]
    boxes, conf, cls = boxes[k], conf[k], cls[k]
    boxes[:, [0, 2]] = (boxes[:, [0, 2]] - left) / r; boxes[:, [1, 3]] = (boxes[:, [1, 3]] - top) / r   # deshacer letterbox
    boxes[:, [0, 2]] = boxes[:, [0, 2]].clip(0, W); boxes[:, [1, 3]] = boxes[:, [1, 3]].clip(0, H)
    clases_contadas = d["clases_contadas"]
    contar = np.isin(cls, clases_contadas)
    return dict(conteo=int(contar.sum()),
                detecciones=[dict(clase=cfg["clases"][int(c)], conf=round(float(s), 4), caja=[round(float(v), 1) for v in b])
                             for b, s, c in zip(boxes, conf, cls)])




# Ruta PAD352 (320 + 16 px de borde gris = entrada 352): umbrales en val, test una vez por semilla, exportacion, paridad, script de referencia y casos de prueba
Regla pre-registrada (ACCEPTANCE.md, Adenda 2). Umbrales: minimo MAE en val con NMS 0,5. Modelo elegido: el de la regla original (semilla 43).

## Entorno y datos

In [ ]:
import os, io, json, shutil, subprocess, sys, time, hashlib
from pathlib import Path
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage
subprocess.run("pip install -q ultralytics", shell=True)
from ultralytics import YOLO
from ultralytics.utils.metrics import compute_ap
OUT = Path("/kaggle/working/outputs"); OUT.mkdir(parents=True, exist_ok=True)
SRC = next(Path("/kaggle/input").glob("**/yolo_ds")); DSR = Path("/kaggle/temp/yolo_ds"); shutil.copytree(SRC, DSR)
find = lambda n: next(Path("/kaggle/input").glob(f"**/{n}"))
MODELS = {"E3": 42, "E3s43": 43, "E3s44": 44}; WTS = {m: find(f"{m}_best.pt") for m in MODELS}; CHOSEN = "E3s43"
names = "\n".join(f"  {i}: {n}" for i, n in enumerate(CLASS_NAMES))
(DSR / "data_base.yaml").write_text(f"path: {DSR}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n{names}\n")
sp = pd.read_csv(DSR / "splits.csv", dtype={"dhash": str}); sp["stem"] = sp.name.str.replace(".jpg", "", regex=False); INFO = sp.set_index("stem"); GROUP_OF = INFO.group.to_dict(); PATTERN = INFO.pattern.to_dict()
IM = {s: sorted((DSR / "images" / s).glob("*.jpg")) for s in ("val", "test")}; GT = {s: load_gts(DSR, s) for s in ("val", "test")}
rng = np.random.default_rng(42); SUB = [IM["test"][i] for i in sorted(rng.choice(len(IM["test"]), 300, replace=False))]; G300 = {p.stem: GT["test"][p.stem] for p in SUB}
CONFS = np.round(np.arange(0.05, 0.701, 0.05), 2); IOU = 0.5

## Funciones (prediccion con borde, AP50 propio, emparejamiento, IC por grupos)

In [ ]:
def pad_canvas(bgr):
    r_ = cv2.resize(bgr, (320, 320), interpolation=cv2.INTER_LINEAR)
    return cv2.copyMakeBorder(r_, 16, 16, 16, 16, cv2.BORDER_CONSTANT, value=(114, 114, 114))

def predict_pad(model, paths, conf=0.01, iou=IOU, batch=16):
    out = {}
    for i in range(0, len(paths), batch):
        ch = paths[i:i + batch]
        for p, r in zip(ch, model.predict([pad_canvas(cv2.imread(str(p))) for p in ch], imgsz=352, conf=conf, iou=iou, verbose=False, device="cpu", max_det=1000)):
            b = r.boxes; out[Path(p).stem] = (b.cls.cpu().numpy().astype(int), b.conf.cpu().numpy(), np.clip((b.xyxy.cpu().numpy() - 16) / 320, 0, 1))
    return out

def ap50(dets, gts, cls=GB):
    rows, ngt = [], 0
    for k, (gc, gb) in gts.items():
        c, s, b = dets[k]; p = np.where(c == cls)[0]; g = np.where(gc == cls)[0]; ngt += len(g); order = p[np.argsort(-s[p])]; iou = box_iou(b[order], gb[g]); used = set()
        for i, pi in enumerate(order):
            best, bv = -1, 0.5
            for j in range(len(g)):
                if j not in used and iou[i, j] >= bv: best, bv = j, iou[i, j]
            if best >= 0: used.add(best)
            rows.append((s[pi], int(best >= 0)))
    rows.sort(key=lambda r: -r[0]); tp = np.cumsum([r[1] for r in rows]); fp = np.cumsum([1 - r[1] for r in rows]); return float(compute_ap(tp / max(ngt, 1), tp / np.maximum(tp + fp, 1))[0])

def match_boxes(dets, gts, conf, cls=GB, thr=0.5):
    rows = []
    for k, (gc, gbx) in gts.items():
        c, s, b = dets[k]; m = s >= conf; pc, pb = c[m], b[m]; g = np.where(gc == cls)[0]; p = np.where(pc == cls)[0]; iou = box_iou(pb[p], gbx[g]); up, ug = set(), set()
        for v, i, j in sorted(((iou[i, j], i, j) for i in range(len(p)) for j in range(len(g))), reverse=True):
            if v < thr: break
            if i in up or j in ug: continue
            up.add(i); ug.add(j)
        rows.append((k, len(up), len(p) - len(up), len(g) - len(ug), len(g), len(p)))
    return pd.DataFrame(rows, columns=["stem", "tp", "fp", "fn", "n_gt", "n_pred"]).set_index("stem")

def summ(mb):
    tp, fp, fn = mb.tp.sum(), mb.fp.sum(), mb.fn.sum(); P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
    return dict(n_img=len(mb), n_gt=int(mb.n_gt.sum()), precision=float(P), recall=float(R), f1=float(2 * P * R / max(P + R, 1e-9)), mae=float((mb.n_pred - mb.n_gt).abs().mean()), bias=float((mb.n_pred - mb.n_gt).mean()))

def boot_groups(mb, n=2000, seed=0):
    g = mb.index.map(GROUP_OF).to_numpy(); ug, inv = np.unique(g, return_inverse=True)
    agg = pd.DataFrame({"tp": mb.tp.to_numpy(), "fp": mb.fp.to_numpy(), "fn": mb.fn.to_numpy(), "ae": (mb.n_pred - mb.n_gt).abs().to_numpy(), "one": 1}).groupby(inv).sum().to_numpy(); rg = np.random.default_rng(seed); res = []
    for _ in range(n):
        t = agg[rg.integers(0, len(ug), len(ug))].sum(0); P = t[0] / max(t[0] + t[1], 1); R = t[0] / max(t[0] + t[2], 1); res.append((P, R, 2 * P * R / max(P + R, 1e-9), t[3] / t[4]))
    q = np.percentile(np.array(res), [2.5, 97.5], axis=0); return {k: [float(q[0, i]), float(q[1, i])] for i, k in enumerate(["precision", "recall", "f1", "mae"])}

def confusion(dets, gts, conf, thr=0.5):
    M = np.zeros((5, 5), int)
    for k, (gc, gbx) in gts.items():
        c, s, b = dets[k]; m = s >= conf; pc, pb = c[m], b[m]; iou = box_iou(gbx, pb); ug, up = set(), set()
        for v, i, j in sorted(((iou[i, j], i, j) for i in range(len(gc)) for j in range(len(pc))), reverse=True):
            if v < thr: break
            if i in ug or j in up: continue
            ug.add(i); up.add(j); M[gc[i], pc[j]] += 1
        for i in range(len(gc)):
            if i not in ug: M[gc[i], 4] += 1
        for j in range(len(pc)):
            if j not in up: M[4, pc[j]] += 1
    return pd.DataFrame(M, index=CLASS_NAMES + ["fondo"], columns=CLASS_NAMES + ["fondo"])

## Paso A. Por semilla: umbrales en VAL (minimo MAE, NMS 0,5) y TEST una vez

In [ ]:
R = {}; DT = {}
for m, sd in MODELS.items():
    mod = YOLO(str(WTS[m])); dv = predict_pad(mod, IM["val"]); fz = {}
    for nm, cl in (("estricta", [GB]), ("amplia", [GB, GPB])):
        g = pd.DataFrame([dict(conf=float(c), mae=count_metrics(dv, GT["val"], c, cl)["mae"]) for c in CONFS]); fz[nm] = dict(conf=float(g.loc[g.mae.idxmin(), "conf"]), mae_val=float(g.mae.min()), iou_nms=IOU)
    fz["val_estricta"] = summ(match_boxes(dv, GT["val"], fz["estricta"]["conf"])); fz["val_ap50"] = ap50(dv, GT["val"])
    dt = predict_pad(mod, IM["test"]); DT[m] = dt
    te = summ(match_boxes(dt, GT["test"], fz["estricta"]["conf"])); te["ap50_propio"] = ap50(dt, GT["test"]); te["mae_amplia"] = count_metrics(dt, GT["test"], fz["amplia"]["conf"], [GB, GPB])["mae"]
    R[m] = dict(seed=sd, thr=fz, test=te); print(m, json.dumps(R[m], indent=1), flush=True)
agg = {k: dict(mean=float(np.mean([R[m]["test"][k] for m in R])), sd=float(np.std([R[m]["test"][k] for m in R], ddof=1))) for k in ("precision", "recall", "f1", "mae", "ap50_propio")}
fz = R[CHOSEN]["thr"]; dets = DT[CHOSEN]; mb = match_boxes(dets, GT["test"], fz["estricta"]["conf"])
detail = dict(modelo=CHOSEN, ruta="PAD352", umbrales=fz, test=R[CHOSEN]["test"], ic95_grupos=boot_groups(mb), por_semilla_agregado=agg, todas_las_semillas=R)
for pat in ("crop", "simple"): sub = mb[mb.index.map(PATTERN) == pat]; detail[f"patron_{pat}"] = dict(**summ(sub), ic95=boot_groups(sub))
cm = confusion(dets, GT["test"], fz["estricta"]["conf"]); cm.to_csv(OUT / "confusion_test_pad352.csv"); print(cm.to_string())
tr_med = float(INFO[INFO.split == "train"][["n_c2"]].median().iloc[0]); yt = np.array([int((GT["test"][k][0] == GB).sum()) for k in GT["test"]]); detail["baseline_mediana_test_mae"] = float(np.abs(yt - tr_med).mean())
(OUT / "test_final_pad352.json").write_text(json.dumps(detail, indent=1, default=float)); print(json.dumps({k: detail[k] for k in ("ic95_grupos", "por_semilla_agregado", "patron_crop", "patron_simple")}, indent=1, default=float))

## Paso B. Exportacion a 352 del modelo elegido: ONNX, TFLite FP32, FP16 (float_casting) e INT8 (calibracion 300 imagenes de train)

In [ ]:
EX = Path("/kaggle/working/export352"); EX.mkdir(exist_ok=True); W = EX / f"ecoli_gb_{CHOSEN}_352.pt"; shutil.copy2(WTS[CHOSEN], W)
tr = INFO[INFO.split == "train"].copy(); tr["qg"] = pd.qcut(tr.gray.rank(method="first"), 4, labels=False); tr["qb"] = pd.qcut(tr.lab_b.rank(method="first"), 4, labels=False)
cal = pd.concat([g.sample(min(len(g), 19), random_state=42) for _, g in tr.groupby(["qg", "qb"])]).head(300)
(EX / "calib.txt").write_text("\n".join(str(DSR / "images" / "train" / n) for n in cal.name)); (EX / "calib.yaml").write_text(f"path: {DSR}\ntrain: {EX/'calib.txt'}\nval: {EX/'calib.txt'}\nnames:\n{names}\n")
def run_export(tag, kw, timeout=1500):
    code = f"from ultralytics import YOLO; print('EXPORT OK', YOLO('{W}').export(imgsz=352, {kw}))"; t = time.time()
    try:
        r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True, timeout=timeout, cwd=str(EX)); print(f"[{tag}] rc={r.returncode} {time.time()-t:.0f}s", (r.stdout + r.stderr)[-300:], flush=True)
    except subprocess.TimeoutExpired: print(f"[{tag}] TIMEOUT", flush=True)
run_export("onnx", "format='onnx', simplify=True"); run_export("tflite fp32", "format='tflite'")
shutil.copy2(EX / f"ecoli_gb_{CHOSEN}_352.tflite", EX / f"ecoli_gb_{CHOSEN}_352_fp32.tflite")
run_export("tflite int8", f"format='tflite', int8=True, data='{EX / 'calib.yaml'}'")
from ai_edge_quantizer import quantizer, qtyping
q = quantizer.Quantizer(str(EX / f"ecoli_gb_{CHOSEN}_352_fp32.tflite"))
q.update_quantization_recipe(regex=".*", operation_name=qtyping.TFLOperationName.ALL_SUPPORTED, algorithm_key="float_casting",
    op_config=qtyping.OpQuantizationConfig(weight_tensor_config=qtyping.TensorQuantizationConfig(num_bits=16, dtype=qtyping.TensorDataType.FLOAT), compute_precision=qtyping.ComputePrecision.FLOAT, explicit_dequantize=True))
q.quantize().export_model(str(EX / f"ecoli_gb_{CHOSEN}_352_fp16.tflite"), overwrite=True)
print(sorted(p.name for p in EX.glob("*")))

## Paso C. Config de la app y script de referencia; paridad en 300 imagenes de test (pt con borde vs TFLite con el script de referencia)

In [ ]:
APP = {"modelo": f"ecoli_gb_{CHOSEN}_352_fp16.tflite", "version": "2.0 (ruta con borde)", "clases": CLASS_NAMES, "max_det": 1000,
       "entrada": {"tamano": 352, "contenido": 320, "borde_gris_px": 16, "forma": [1, 3, 352, 352], "orden": "NCHW", "canales": "RGB", "dtype": "float32", "normalizacion": "pixel/255, sin media ni desviacion",
                   "preprocesado": "1) letterbox a 320 (lado mayor=320, relleno gris 114 centrado, bilineal); 2) agregar 16 px de gris 114 por cada lado -> 352x352"},
       "salida": {"forma": [1, 8, 2541], "filas_0_3": "cx,cy,w,h normalizados 0-1 respecto a 352", "filas_4_7": "score por clase 0-1 (sin sigmoide)", "nms_incluido": False},
       "definiciones": {"estricta": {"clases_contadas": [2], "conf": fz["estricta"]["conf"], "iou_nms": IOU}, "amplia": {"clases_contadas": [2, 3], "conf": fz["amplia"]["conf"], "iou_nms": IOU}},
       "definicion_recomendada": "estricta", "nms": "por clase; se filtra con max(score de clases) > conf; se cuenta solo si la clase ganadora esta en clases_contadas"}
(OUT / "app_config.json").write_text(json.dumps(APP, indent=1, ensure_ascii=False)); open(OUT / "inferir_tflite.py", "w", encoding="utf8").write(INFERIR_SRC)
TFF = {k: EX / f"ecoli_gb_{CHOSEN}_352{s}.tflite" for k, s in (("fp32", "_fp32"), ("fp16", "_fp16"), ("int8", "_int8"))}
i8 = sorted(EX.glob("*int8*.tflite")); TFF["int8"] = i8[0] if i8 else TFF["int8"]; print(TFF)
def script_dets(path, paths, conf_floor=0.01):
    it = cargar_interprete(str(path)); cfg = json.loads(json.dumps(APP)); cfg["definiciones"]["todas"] = {"clases_contadas": [0, 1, 2, 3], "conf": conf_floor, "iou_nms": IOU}; out = {}
    for p in paths:
        d = detectar(it, cv2.imread(str(p)), cfg, "todas")["detecciones"]
        out[p.stem] = (np.array([CLASS_NAMES.index(x["clase"]) for x in d], dtype=int), np.array([x["conf"] for x in d]), np.array([x["caja"] for x in d]).reshape(-1, 4) / 640)
    return out
par = {"pt_borde": {}, **{f"script_tflite_{k}": {} for k in TFF}}; ptm = YOLO(str(WTS[CHOSEN])); cth = fz["estricta"]["conf"]
d = predict_pad(ptm, SUB); par["pt_borde"] = dict(ap50_propio=ap50(d, G300), **summ(match_boxes(d, G300, cth)))
for k, f in TFF.items():
    if not Path(f).exists(): par[f"script_tflite_{k}"] = dict(error="no existe"); continue
    t0 = time.time(); d = script_dets(f, SUB); par[f"script_tflite_{k}"] = dict(ap50_propio=ap50(d, G300), **summ(match_boxes(d, G300, cth)), seg_300_imgs=round(time.time() - t0, 1), mb=round(Path(f).stat().st_size / 1e6, 2))
(OUT / "paridad_300_pad352.json").write_text(json.dumps(par, indent=1)); print(json.dumps(par, indent=1))
import tensorflow as tf
io_ = {}
for k, f in TFF.items():
    if Path(f).exists():
        it_ = tf.lite.Interpreter(model_path=str(f)); it_.allocate_tensors(); io_[k] = dict(inputs=[dict(shape=x["shape"].tolist(), dtype=x["dtype"].__name__, quant=[float(x["quantization"][0]), int(x["quantization"][1])]) for x in it_.get_input_details()],
                                                                                       outputs=[dict(shape=x["shape"].tolist(), dtype=x["dtype"].__name__, quant=[float(x["quantization"][0]), int(x["quantization"][1])]) for x in it_.get_output_details()])
(OUT / "tflite_io_352.json").write_text(json.dumps(io_, indent=1)); print(json.dumps(io_)[:900])

## Paso D. Verificacion del script contra ultralytics (misma entrada con borde) y 20 casos de prueba con etiqueta y prediccion impresas

In [ ]:
it = cargar_interprete(str(TFF["fp16"])); uc = YOLO(str(TFF["fp16"]), task="detect"); agree, diffs = 0, []
for p in SUB:
    img = cv2.imread(str(p)); mine = detectar(it, img, APP, "estricta"); r = uc.predict(pad_canvas(img), imgsz=352, conf=cth, iou=IOU, device="cpu", verbose=False, max_det=1000)[0]
    sel = r.boxes.cls.cpu().numpy() == GB; bu = np.clip((r.boxes.xyxy.cpu().numpy()[sel] - 16) / 320 * 640, 0, 640); bm = np.array([x["caja"] for x in mine["detecciones"] if x["clase"] == "G-bacilli"]).reshape(-1, 4)
    agree += int(int(sel.sum()) == mine["conteo"])
    if len(bu) == len(bm) and len(bu): diffs.append(float(np.abs(np.sort(bu, 0) - np.sort(bm, 0)).max()))
ver = dict(imagenes=len(SUB), conteo_igual_a_ultralytics=agree, max_dif_caja_px=float(max(diffs)) if diffs else None, mediana_dif_caja_px=float(np.median(diffs)) if diffs else None); print("script vs ultralytics:", ver)
CAS = OUT / "casos"; CAS.mkdir(exist_ok=True); rs = np.random.default_rng(7); stems = [p.stem for p in IM["test"]]; ngt = {s: int((GT["test"][s][0] == GB).sum()) for s in stems}; pick = []
for lo, hi, n in ((0, 0, 4), (1, 1, 6), (2, 3, 5), (4, 99, 5)): pick += list(rs.choice([s for s in stems if lo <= ngt[s] <= hi], n, replace=False))
def panel(img_bgr, gc, gb, dets):
    gt_v = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).copy(); pr_v = gt_v.copy()
    for c, b in zip(gc, gb):
        x1, y1, x2, y2 = (b * 640).astype(int); cv2.rectangle(gt_v, (x1, y1), (x2, y2), (0, 200, 0) if c == GB else (255, 160, 0), 2)
    for d in dets:
        x1, y1, x2, y2 = [int(v) for v in d["caja"]]; col = (255, 0, 0) if d["clase"] == "G-bacilli" else (150, 0, 255); cv2.rectangle(pr_v, (x1, y1), (x2, y2), col, 2); cv2.putText(pr_v, f'{d["conf"]:.2f}', (x1, max(y1 - 6, 14)), 0, 0.6, col, 2)
    return gt_v, pr_v
print("Leyenda: verde = G-bacilli real (etiqueta); naranja = otra clase real; rojo = prediccion G-bacilli (con confianza); violeta = prediccion de otra clase")
casos = []
for n_, s in enumerate(pick, 1):
    img = cv2.imread(str(DSR / "images" / "test" / f"{s}.jpg")); cv2.imwrite(str(CAS / f"{s}.jpg"), img); res_ = detectar(it, img, APP, "estricta"); res_a = detectar(it, img, APP, "amplia"); gc, gb = GT["test"][s]
    gtb = [[round(float(v), 1) for v in (b * 640)] for c, b in zip(gc, gb) if c == GB]; dg = [d for d in res_["detecciones"] if d["clase"] == "G-bacilli"]
    iou_ = box_iou(np.array([d["caja"] for d in dg]).reshape(-1, 4) / 640, np.array(gtb).reshape(-1, 4) / 640); tp_ = len(set(np.where(iou_ >= 0.5)[0])) if iou_.size else 0
    casos.append(dict(imagen=f"{s}.jpg", patron=INFO.pattern[s], gt_G_bacilli=len(gtb), gt_cajas_G_bacilli=gtb, esperado_script_estricta=res_["conteo"], esperado_script_amplia=res_a["conteo"], detecciones_estricta=dg))
    print(f"[{n_:02d}/20] {s}.jpg ({INFO.pattern[s]}) | etiqueta real G-bacilli = {len(gtb)} | prediccion del modelo = {res_['conteo']} (confianzas {[d['conf'] for d in dg]}) | aciertos~{tp_} falsos+~{len(dg) - tp_} perdidos~{len(gtb) - tp_}")
    gt_v, pr_v = panel(img, gc, gb, res_["detecciones"]); fig, ax = plt.subplots(1, 2, figsize=(12, 6.2)); ax[0].imshow(gt_v); ax[0].set_title(f"ETIQUETA REAL: {len(gtb)} G-bacilli", fontsize=13); ax[1].imshow(pr_v); ax[1].set_title(f"PREDICCION DEL MODELO: {res_['conteo']} G-bacilli", fontsize=13)
    for x in ax: x.axis("off")
    plt.tight_layout(); buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=70); plt.close(fig); display(IPImage(data=buf.getvalue())); (CAS / f"panel_{n_:02d}_{s}.png").write_bytes(buf.getvalue())
nosq = []
for k, s in enumerate(pick[4:7]):
    full = cv2.imread(str(DSR / "images" / "test" / f"{s}.jpg")); y0, y1, x0, x1 = ((80, 560, 0, 640), (0, 640, 100, 500), (160, 400, 20, 600))[k]; crop = full[y0:y1, x0:x1]
    cv2.imwrite(str(CAS / f"nocuadrada_{k}_{s}.jpg"), crop); mc = detectar(it, crop, APP, "estricta"); mf = detectar(it, full, APP, "estricta")
    inside = [d for d in mf["detecciones"] if d["clase"] == "G-bacilli" and d["caja"][0] > x0 + 15 and d["caja"][2] < x1 - 15 and d["caja"][1] > y0 + 15 and d["caja"][3] < y1 - 15]
    bc = np.array([[d["caja"][0] + x0, d["caja"][1] + y0, d["caja"][2] + x0, d["caja"][3] + y0] for d in mc["detecciones"] if d["clase"] == "G-bacilli"]).reshape(-1, 4); bf = np.array([d["caja"] for d in inside]).reshape(-1, 4)
    iou_ = box_iou(bf, bc) if len(bf) and len(bc) else np.zeros((0, 0))
    nosq.append(dict(imagen=f"nocuadrada_{k}_{s}.jpg", forma=list(crop.shape[:2]), conteo_recorte=mc["conteo"], cajas_en_imagen_completa_dentro_del_recorte=int(len(bf)), cajas_del_recorte_con_IoU_mayor_0_7_con_la_completa=int((iou_.max(1) > 0.7).sum()) if iou_.size else 0, detecciones_recorte=mc["detecciones"]))
(OUT / "casos_prueba.json").write_text(json.dumps(dict(config="app_config.json", modelo=APP["modelo"], verificacion_vs_ultralytics_300=ver, casos=casos, no_cuadradas=nosq), indent=1, ensure_ascii=False))
tab = pd.DataFrame([(c["imagen"], c["patron"], c["gt_G_bacilli"], c["esperado_script_estricta"], c["esperado_script_amplia"]) for c in casos], columns=["imagen", "patron", "GT", "modelo_estricta", "modelo_amplia"]); tab["error"] = tab.modelo_estricta - tab.GT
tab.to_csv(OUT / "casos_prueba_tabla.csv", index=False); print(tab.to_string()); print("MAE en estos 20:", tab.error.abs().mean()); print("no cuadradas:", json.dumps([{k: v for k, v in n.items() if k != "detecciones_recorte"} for n in nosq]))
print("script vs ultralytics:", ver)
for f in EX.glob("ecoli_gb_*"):
    if f.suffix in (".pt", ".onnx", ".tflite"): shutil.copy2(f, OUT / f.name)
(OUT / "sha256_pesos_352.json").write_text(json.dumps({p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.glob("ecoli_gb_*") if p.suffix in (".pt", ".onnx", ".tflite")}, indent=1))
(OUT / "pip_freeze_final.txt").write_text(subprocess.run("pip freeze | grep -i -E '^(ultralytics|torch|tensorflow|onnx|onnx2tf|onnxruntime|numpy|ai-edge|opencv|scikit|scipy|pandas)'", shell=True, capture_output=True, text=True).stdout)